# 🛠️ Data Preparation — Credit Score Classification

This notebook prepares the raw Credit Score dataset for machine learning.  
All steps are implemented as **reusable functions** that are applied consistently to train, validation, and test sets.

### Pipeline Overview
1. [Load Data & Reproduce Splits](#1)
2. [Drop PII & Non-Informative Columns](#2)
3. [Clean Numeric Columns](#3)
4. [Cap Outliers](#4)
5. [Parse Credit History Age](#5)
6. [Clean Categorical Columns](#6)
7. [Impute Missing Values](#7)
8. [Encode Categorical Features](#8)
9. [Encode Target Variable](#9)
10. [Final Validation](#10)
11. [Export Prepared Data](#11)

## 1. Load Data & Reproduce Splits <a id='1'></a>

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
def load_data(train_path: str, test_path: str):
    """
    Load raw train and test CSVs.
    Reproduces the 80/20 stratified split from EDA to get labeled train and validation sets.

    Returns
    -------
    train_set : pd.DataFrame  — 80,000 rows, labeled
    valid_set : pd.DataFrame  — 20,000 rows, labeled
    test_set  : pd.DataFrame  — 50,000 rows, unlabeled (Kaggle inference)
    """
    train_raw = pd.read_csv(train_path, low_memory=False)
    test_set  = pd.read_csv(test_path,  low_memory=False)

    train_set, valid_set = train_test_split(
        train_raw,
        test_size=0.2,
        random_state=42,
        stratify=train_raw['Credit_Score'],
        shuffle=True
    )

    train_set = train_set.reset_index(drop=True)
    valid_set = valid_set.reset_index(drop=True)
    test_set  = test_set.reset_index(drop=True)

    print(f'Train set  : {train_set.shape}')
    print(f'Valid set  : {valid_set.shape}')
    print(f'Test set   : {test_set.shape}')

    return train_set, valid_set, test_set


train_set, valid_set, test_set = load_data(
    r"../data/raw/train_raw.csv",
    r"../data/raw/test_raw.csv"
)

Train set  : (80000, 28)
Valid set  : (20000, 28)
Test set   : (50000, 27)


## 2. Drop PII & Non-Informative Columns <a id='2'></a>

The following columns are dropped before any modelling:

| Column | Reason |
|---|---|
| `ID` | Row identifier — no predictive value |
| `Customer_ID` | Customer identifier — no predictive value |
| `Name` | PII — personally identifiable, not useful |
| `SSN` | PII — sensitive, heavily corrupted, not useful |
| `Month` | Temporal identifier — customers appear across 8 months; leaks time ordering |

In [3]:
COLUMNS_TO_DROP = ['ID', 'Customer_ID', 'Name', 'SSN', 'Month']

def drop_pii_columns(df: pd.DataFrame, columns: list = COLUMNS_TO_DROP) -> pd.DataFrame:
    """
    Drop PII and non-informative columns from the dataframe.

    Parameters
    ----------
    df      : Input dataframe
    columns : List of column names to drop

    Returns
    -------
    df : Dataframe with specified columns removed
    """
    cols_present = [c for c in columns if c in df.columns]
    df = df.drop(columns=cols_present)
    print(f'Dropped {len(cols_present)} columns: {cols_present}')
    print(f'Remaining columns: {df.shape[1]}')
    return df


train_set = drop_pii_columns(train_set)
valid_set = drop_pii_columns(valid_set)
test_set  = drop_pii_columns(test_set)

Dropped 5 columns: ['ID', 'Customer_ID', 'Name', 'SSN', 'Month']
Remaining columns: 23
Dropped 5 columns: ['ID', 'Customer_ID', 'Name', 'SSN', 'Month']
Remaining columns: 23
Dropped 5 columns: ['ID', 'Customer_ID', 'Name', 'SSN', 'Month']
Remaining columns: 22


## 3. Clean Numeric Columns <a id='3'></a>

Many numeric columns were stored as strings with embedded garbage characters.  
Each is stripped and cast to a proper numeric type. Invalid values become `NaN` for imputation later.

| Column | Issue | Fix |
|---|---|---|
| `Age` | Trailing chars (`28_`), negatives (`-500`), values > 100 | Strip non-digits, clip to [18, 100] |
| `Annual_Income` | Trailing underscores (`34847.84_`) | Strip `_`, cast float |
| `Num_of_Loan` | Trailing chars, negatives, extreme values (`967`) | Strip, cast int, clip to [0, 10] |
| `Num_of_Delayed_Payment` | Trailing chars (`8_`), negatives | Strip, cast float, clip to [0, ∞) |
| `Outstanding_Debt` | Trailing underscores | Strip `_`, cast float |
| `Changed_Credit_Limit` | Placeholder `_`, negatives | Replace `_` → NaN, cast float |
| `Amount_invested_monthly` | Garbage placeholder `__10000__` | Replace → NaN, cast float |
| `Monthly_Balance` | Garbage placeholder `__-333333...___` | Replace → NaN, cast float |

In [4]:
def clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean all numeric columns that contain embedded garbage characters,
    invalid placeholders, or out-of-range values.

    Steps per column:
      - Strip non-numeric characters (where applicable)
      - Replace known garbage placeholders with NaN
      - Cast to appropriate numeric dtype
      - Clip values to valid domain ranges

    Parameters
    ----------
    df : Input dataframe

    Returns
    -------
    df : Dataframe with cleaned numeric columns
    """
    df = df.copy()

    # Age: strip non-digit chars, cast int, clip to valid human age range
    df['Age'] = (
        df['Age'].astype(str)
                 .str.replace(r'[^\d]', '', regex=True)
    )
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df['Age'] = df['Age'].where(df['Age'].between(18, 100))

    # Annual_Income: strip trailing underscores
    df['Annual_Income'] = pd.to_numeric(
        df['Annual_Income'].astype(str).str.replace('_', '', regex=False),
        errors='coerce'
    )

    # Num_of_Loan: strip underscores, cast int, clip to [0, 10]
    df['Num_of_Loan'] = pd.to_numeric(
        df['Num_of_Loan'].astype(str).str.replace('_', '', regex=False),
        errors='coerce'
    )
    df['Num_of_Loan'] = df['Num_of_Loan'].clip(lower=0, upper=10)

    # Num_of_Delayed_Payment: strip underscores, cast float, clip negatives
    df['Num_of_Delayed_Payment'] = pd.to_numeric(
        df['Num_of_Delayed_Payment'].astype(str).str.replace('_', '', regex=False),
        errors='coerce'
    )
    df['Num_of_Delayed_Payment'] = df['Num_of_Delayed_Payment'].clip(lower=0)

    # Outstanding_Debt: strip trailing underscores
    df['Outstanding_Debt'] = pd.to_numeric(
        df['Outstanding_Debt'].astype(str).str.replace('_', '', regex=False),
        errors='coerce'
    )

    # Changed_Credit_Limit: replace placeholder '_' with NaN
    df['Changed_Credit_Limit'] = df['Changed_Credit_Limit'].replace('_', np.nan)
    df['Changed_Credit_Limit'] = pd.to_numeric(df['Changed_Credit_Limit'], errors='coerce')

    # Amount_invested_monthly: replace garbage placeholder, cast float
    df['Amount_invested_monthly'] = df['Amount_invested_monthly'].replace('__10000__', np.nan)
    df['Amount_invested_monthly'] = pd.to_numeric(df['Amount_invested_monthly'], errors='coerce')

    # Monthly_Balance: replace garbage placeholder, cast float
    df['Monthly_Balance'] = df['Monthly_Balance'].replace(
        '__-333333333333333333333333333__', np.nan
    )
    df['Monthly_Balance'] = pd.to_numeric(df['Monthly_Balance'], errors='coerce')

    # Delay_from_due_date: clip negatives to 0
    df['Delay_from_due_date'] = df['Delay_from_due_date'].clip(lower=0)

    return df


train_set = clean_numeric_columns(train_set)
valid_set = clean_numeric_columns(valid_set)
test_set  = clean_numeric_columns(test_set)

print('\nDtypes after cleaning:')
print(train_set[['Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
                  'Outstanding_Debt', 'Changed_Credit_Limit',
                  'Amount_invested_monthly', 'Monthly_Balance']].dtypes)


Dtypes after cleaning:
Age                        float64
Annual_Income              float64
Num_of_Loan                  int64
Num_of_Delayed_Payment     float64
Outstanding_Debt           float64
Changed_Credit_Limit       float64
Amount_invested_monthly    float64
Monthly_Balance            float64
dtype: object


## 4. Cap Outliers <a id='4'></a>

Several integer columns have extreme values that are clearly data entry errors.  
We cap them at realistic domain limits rather than removing rows.

| Column | Issue | Cap |
|---|---|---|
| `Num_Bank_Accounts` | Negatives, max=1798 | Clip to [0, 20] |
| `Num_Credit_Card` | Max=1499 | Clip to [0, 20] |
| `Interest_Rate` | Max=5797 | Clip to [1, 100] |
| `Num_Credit_Inquiries` | Max=2597 | Clip to [0, 20] |
| `Annual_Income` | Max=24M (likely entry error) | Clip at 99th percentile |

In [5]:
def cap_outliers(df: pd.DataFrame, annual_income_cap: float = None) -> pd.DataFrame:
    """
    Cap outliers in columns where extreme values are clearly data entry errors.
    Hard limits are applied based on domain knowledge.
    Annual_Income is capped at the 99th percentile of the training set.

    Parameters
    ----------
    df                : Input dataframe
    annual_income_cap : 99th percentile cap value (computed from train, applied to all splits)

    Returns
    -------
    df                : Dataframe with capped columns
    annual_income_cap : The cap value (pass to valid/test to ensure consistency)
    """
    df = df.copy()

    df['Num_Bank_Accounts']   = df['Num_Bank_Accounts'].clip(lower=0, upper=20)
    df['Num_Credit_Card']     = df['Num_Credit_Card'].clip(lower=0, upper=20)
    df['Interest_Rate']       = df['Interest_Rate'].clip(lower=1, upper=100)
    df['Num_Credit_Inquiries']= df['Num_Credit_Inquiries'].clip(lower=0, upper=20)

    # Annual_Income: compute cap from train set only, reuse for valid/test
    if annual_income_cap is None:
        annual_income_cap = df['Annual_Income'].quantile(0.99)
        print(f'Annual_Income 99th percentile cap: {annual_income_cap:,.2f}')
    df['Annual_Income'] = df['Annual_Income'].clip(upper=annual_income_cap)

    return df, annual_income_cap


train_set, income_cap = cap_outliers(train_set)
valid_set, _          = cap_outliers(valid_set, annual_income_cap=income_cap)
test_set,  _          = cap_outliers(test_set,  annual_income_cap=income_cap)

print(f'\nNum_Bank_Accounts  — max after cap: {train_set["Num_Bank_Accounts"].max()}')
print(f'Num_Credit_Card    — max after cap: {train_set["Num_Credit_Card"].max()}')
print(f'Interest_Rate      — max after cap: {train_set["Interest_Rate"].max()}')
print(f'Num_Credit_Inquiries — max after cap: {train_set["Num_Credit_Inquiries"].max()}')

Annual_Income 99th percentile cap: 179,825.36

Num_Bank_Accounts  — max after cap: 20
Num_Credit_Card    — max after cap: 20
Interest_Rate      — max after cap: 100
Num_Credit_Inquiries — max after cap: 20.0


## 5. Parse Credit History Age <a id='5'></a>

`Credit_History_Age` is stored as a human-readable string like `"22 Years and 9 Months"`.  
We convert this to a single integer: **total number of months**.

In [6]:
def parse_credit_history_age(df: pd.DataFrame) -> pd.DataFrame:
    """
    Parse the 'Credit_History_Age' string column into a numeric feature
    representing total credit history in months.

    Example: '22 Years and 9 Months' → 273

    Parameters
    ----------
    df : Input dataframe

    Returns
    -------
    df : Dataframe with 'Credit_History_Age' replaced by integer months
    """
    def _parse(val):
        if pd.isna(val):
            return np.nan
        match = re.search(r'(\d+)\s+Years?\s+and\s+(\d+)\s+Months?', str(val))
        if match:
            return int(match.group(1)) * 12 + int(match.group(2))
        return np.nan

    df = df.copy()
    df['Credit_History_Age'] = df['Credit_History_Age'].apply(_parse)
    print(f'Credit_History_Age — dtype: {df["Credit_History_Age"].dtype}')
    print(f'  min: {df["Credit_History_Age"].min():.0f} months | '
          f'max: {df["Credit_History_Age"].max():.0f} months | '
          f'NaN: {df["Credit_History_Age"].isna().sum()}')
    return df


train_set = parse_credit_history_age(train_set)
valid_set = parse_credit_history_age(valid_set)
test_set  = parse_credit_history_age(test_set)

Credit_History_Age — dtype: float64
  min: 1 months | max: 404 months | NaN: 7240
Credit_History_Age — dtype: float64
  min: 2 months | max: 404 months | NaN: 1790
Credit_History_Age — dtype: float64
  min: 10 months | max: 408 months | NaN: 4470


## 6. Clean Categorical Columns <a id='6'></a>

Categorical columns contain placeholder/garbage values that must be replaced with `NaN` before imputation.

| Column | Garbage Value | Action |
|---|---|---|
| `Occupation` | `_______` | → NaN |
| `Credit_Mix` | `_` | → NaN |
| `Payment_Behaviour` | `!@9#%8` | → NaN |
| `Payment_of_Min_Amount` | `NM` | → NaN |
| `Type_of_Loan` | Already NaN in source | Leave as-is |

In [7]:
def clean_categorical_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace known garbage and placeholder values in categorical columns with NaN.

    Parameters
    ----------
    df : Input dataframe

    Returns
    -------
    df : Dataframe with cleaned categorical columns
    """
    df = df.copy()

    df['Occupation']             = df['Occupation'].replace('_______', np.nan)
    df['Credit_Mix']             = df['Credit_Mix'].replace('_', np.nan)
    df['Payment_Behaviour']      = df['Payment_Behaviour'].replace('!@9#%8', np.nan)
    df['Payment_of_Min_Amount']  = df['Payment_of_Min_Amount'].replace('NM', np.nan)

    print('NaN counts after categorical cleaning:')
    for col in ['Occupation', 'Credit_Mix', 'Payment_Behaviour', 'Payment_of_Min_Amount']:
        print(f'  {col:<30}: {df[col].isna().sum():,}')
    return df


train_set = clean_categorical_columns(train_set)
valid_set = clean_categorical_columns(valid_set)
test_set  = clean_categorical_columns(test_set)

NaN counts after categorical cleaning:
  Occupation                    : 5,691
  Credit_Mix                    : 16,108
  Payment_Behaviour             : 6,098
  Payment_of_Min_Amount         : 9,634
NaN counts after categorical cleaning:
  Occupation                    : 1,371
  Credit_Mix                    : 4,087
  Payment_Behaviour             : 1,502
  Payment_of_Min_Amount         : 2,373
NaN counts after categorical cleaning:
  Occupation                    : 3,438
  Credit_Mix                    : 9,805
  Payment_Behaviour             : 3,800
  Payment_of_Min_Amount         : 5,993


## 7. Impute Missing Values <a id='7'></a>

**Strategy:**
- **Numeric columns** → median imputation (robust to remaining outliers)
- **Categorical columns** → mode imputation
- All imputation statistics are **computed from the train set only** and applied to valid and test to prevent data leakage.

In [8]:
NUMERIC_COLS = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
    'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',
    'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
    'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age',
    'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance'
]

CATEGORICAL_COLS = [
    'Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour'
]


def compute_imputation_stats(train_df: pd.DataFrame) -> dict:
    """
    Compute median for numeric columns and mode for categorical columns
    from the training set only.

    Parameters
    ----------
    train_df : Training dataframe

    Returns
    -------
    stats : dict with keys 'numeric' and 'categorical'
    """
    stats = {
        'numeric':     {col: train_df[col].median() for col in NUMERIC_COLS if col in train_df.columns},
        'categorical': {col: train_df[col].mode()[0] for col in CATEGORICAL_COLS if col in train_df.columns}
    }
    print('Imputation stats computed from train set.')
    print('\nNumeric medians:')
    for k, v in stats['numeric'].items():
        print(f'  {k:<30}: {v:.4f}')
    print('\nCategorical modes:')
    for k, v in stats['categorical'].items():
        print(f'  {k:<30}: {v}')
    return stats


def impute_missing_values(df: pd.DataFrame, stats: dict) -> pd.DataFrame:
    """
    Apply imputation using pre-computed statistics.
    Numeric columns → median fill.
    Categorical columns → mode fill.

    Parameters
    ----------
    df    : Input dataframe
    stats : Dict of imputation values from compute_imputation_stats()

    Returns
    -------
    df : Dataframe with no missing values
    """
    df = df.copy()

    for col, median_val in stats['numeric'].items():
        if col in df.columns:
            df[col] = df[col].fillna(median_val)

    for col, mode_val in stats['categorical'].items():
        if col in df.columns:
            df[col] = df[col].fillna(mode_val)

    remaining_na = df.isnull().sum()
    remaining_na = remaining_na[remaining_na > 0]
    if remaining_na.empty:
        print(f'✅ No missing values remaining.')
    else:
        print(f'⚠️  Remaining NaN after imputation:')
        print(remaining_na)
    return df


# Compute from train only
imputation_stats = compute_imputation_stats(train_set)

# Apply to all splits
print('\n--- Train ---')
train_set = impute_missing_values(train_set, imputation_stats)
print('--- Valid ---')
valid_set = impute_missing_values(valid_set, imputation_stats)
print('--- Test ---')
test_set  = impute_missing_values(test_set,  imputation_stats)

Imputation stats computed from train set.

Numeric medians:
  Age                           : 34.0000
  Annual_Income                 : 37482.3700
  Monthly_Inhand_Salary         : 3086.6833
  Num_Bank_Accounts             : 6.0000
  Num_Credit_Card               : 5.0000
  Interest_Rate                 : 14.0000
  Num_of_Loan                   : 3.0000
  Delay_from_due_date           : 18.0000
  Num_of_Delayed_Payment        : 14.0000
  Changed_Credit_Limit          : 9.4100
  Num_Credit_Inquiries          : 6.0000
  Outstanding_Debt              : 1163.5700
  Credit_Utilization_Ratio      : 32.2926
  Credit_History_Age            : 220.0000
  Total_EMI_per_month           : 68.8397
  Amount_invested_monthly       : 129.0256
  Monthly_Balance               : 336.5681

Categorical modes:
  Occupation                    : Lawyer
  Credit_Mix                    : Standard
  Payment_of_Min_Amount         : Yes
  Payment_Behaviour             : Low_spent_Small_value_payments

--- Train ---

## 8. Encode Categorical Features <a id='8'></a>

| Column | Method | Mapping |
|---|---|---|
| `Credit_Mix` | Ordinal | `Bad`=0, `Standard`=1, `Good`=2 |
| `Payment_of_Min_Amount` | Binary | `No`=0, `Yes`=1 |
| `Occupation` | One-hot encoding | 15 categories → dummy columns |
| `Payment_Behaviour` | One-hot encoding | 6 categories → dummy columns |
| `Type_of_Loan` | Dropped | Free-text multi-label, too sparse to encode cleanly |

In [9]:
CREDIT_MIX_MAP = {'Bad': 0, 'Standard': 1, 'Good': 2}
PAYMENT_MIN_MAP = {'No': 0, 'Yes': 1}


def encode_categorical_features(df: pd.DataFrame, train_columns: list = None) -> pd.DataFrame:
    """
    Encode categorical features using ordinal and one-hot encoding.

    - Credit_Mix          : ordinal (Bad=0, Standard=1, Good=2)
    - Payment_of_Min_Amount: binary  (No=0, Yes=1)
    - Occupation          : one-hot (drop_first=False)
    - Payment_Behaviour   : one-hot (drop_first=False)
    - Type_of_Loan        : dropped (free-text multi-label, too sparse)

    Parameters
    ----------
    df            : Input dataframe
    train_columns : Column list from training set (to align valid/test dummies)

    Returns
    -------
    df            : Encoded dataframe
    train_columns : Final column list (pass this when encoding valid/test)
    """
    df = df.copy()

    # Drop Type_of_Loan
    if 'Type_of_Loan' in df.columns:
        df = df.drop(columns=['Type_of_Loan'])

    # Ordinal: Credit_Mix
    df['Credit_Mix'] = df['Credit_Mix'].map(CREDIT_MIX_MAP)

    # Binary: Payment_of_Min_Amount
    df['Payment_of_Min_Amount'] = df['Payment_of_Min_Amount'].map(PAYMENT_MIN_MAP)

    # One-hot: Occupation and Payment_Behaviour
    df = pd.get_dummies(df, columns=['Occupation', 'Payment_Behaviour'], drop_first=False)

    # Align columns with train set to handle unseen categories in valid/test
    if train_columns is not None:
        target_col = 'Credit_Score' if 'Credit_Score' in df.columns else None
        feature_cols = [c for c in train_columns if c != 'Credit_Score']
        for col in feature_cols:
            if col not in df.columns:
                df[col] = 0
        df = df[[c for c in train_columns if c in df.columns]]
    else:
        train_columns = df.columns.tolist()

    return df, train_columns


train_set, train_cols = encode_categorical_features(train_set)
valid_set, _          = encode_categorical_features(valid_set, train_columns=train_cols)
test_set,  _          = encode_categorical_features(test_set,  train_columns=[c for c in train_cols if c != 'Credit_Score'])

print(f'\nFinal shape — Train : {train_set.shape}')
print(f'Final shape — Valid : {valid_set.shape}')
print(f'Final shape — Test  : {test_set.shape}')
print(f'\nNew columns added by one-hot encoding:')
ohe_cols = [c for c in train_set.columns if c.startswith('Occupation_') or c.startswith('Payment_Behaviour_')]
print(ohe_cols)


Final shape — Train : (80000, 41)
Final shape — Valid : (20000, 41)
Final shape — Test  : (50000, 40)

New columns added by one-hot encoding:
['Occupation_Accountant', 'Occupation_Architect', 'Occupation_Developer', 'Occupation_Doctor', 'Occupation_Engineer', 'Occupation_Entrepreneur', 'Occupation_Journalist', 'Occupation_Lawyer', 'Occupation_Manager', 'Occupation_Mechanic', 'Occupation_Media_Manager', 'Occupation_Musician', 'Occupation_Scientist', 'Occupation_Teacher', 'Occupation_Writer', 'Payment_Behaviour_High_spent_Large_value_payments', 'Payment_Behaviour_High_spent_Medium_value_payments', 'Payment_Behaviour_High_spent_Small_value_payments', 'Payment_Behaviour_Low_spent_Large_value_payments', 'Payment_Behaviour_Low_spent_Medium_value_payments', 'Payment_Behaviour_Low_spent_Small_value_payments']


## 9. Encode Target Variable <a id='9'></a>

Encode `Credit_Score` as an ordinal integer for model training.

| Label | Encoded |
|---|---|
| `Poor` | 0 |
| `Standard` | 1 |
| `Good` | 2 |

In [10]:
CREDIT_SCORE_MAP = {'Poor': 0, 'Standard': 1, 'Good': 2}
CREDIT_SCORE_INVERSE = {v: k for k, v in CREDIT_SCORE_MAP.items()}


def encode_target(df: pd.DataFrame, target_col: str = 'Credit_Score') -> pd.DataFrame:
    """
    Encode the target variable Credit_Score as an ordinal integer.
    Poor=0, Standard=1, Good=2.
    Skipped silently if target column is not present (e.g. test set).

    Parameters
    ----------
    df         : Input dataframe
    target_col : Name of the target column

    Returns
    -------
    df : Dataframe with encoded target column
    """
    if target_col not in df.columns:
        print(f'  {target_col} not found — skipping (test set).')
        return df

    df = df.copy()
    df[target_col] = df[target_col].map(CREDIT_SCORE_MAP)
    print(f'  {target_col} encoded. Value counts:')
    print(df[target_col].value_counts().sort_index().to_string())
    return df


print('--- Train ---')
train_set = encode_target(train_set)
print('--- Valid ---')
valid_set = encode_target(valid_set)
print('--- Test  ---')
test_set  = encode_target(test_set)

--- Train ---
  Credit_Score encoded. Value counts:
Credit_Score
0    23199
1    42539
2    14262
--- Valid ---
  Credit_Score encoded. Value counts:
Credit_Score
0     5799
1    10635
2     3566
--- Test  ---
  Credit_Score not found — skipping (test set).


## 10. Final Validation <a id='10'></a>

Sanity checks before exporting to CSV.

In [11]:
def validate_prepared_data(train_df, valid_df, test_df):
    """
    Run sanity checks on all three prepared splits.
    Checks: no missing values, expected shapes, dtype consistency,
    no remaining garbage strings, column alignment.
    """
    print('=' * 55)
    print('FINAL DATA VALIDATION')
    print('=' * 55)

    for name, df in [('Train', train_df), ('Valid', valid_df), ('Test', test_df)]:
        print(f'\n--- {name} ({df.shape}) ---')

        # Missing values
        na_count = df.isnull().sum().sum()
        status = '✅' if na_count == 0 else '❌'
        print(f'  {status} Missing values       : {na_count}')

        # All numeric
        non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
        status = '✅' if len(non_numeric) == 0 else '⚠️ '
        print(f'  {status} Non-numeric columns   : {non_numeric if non_numeric else "None"}')

        # Negative values in columns that should be non-negative
        non_neg_cols = ['Age', 'Annual_Income', 'Num_Bank_Accounts', 'Num_Credit_Card',
                        'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment']
        neg_issues = [c for c in non_neg_cols if c in df.columns and (df[c] < 0).any()]
        status = '✅' if not neg_issues else '❌'
        print(f'  {status} Negative in non-neg cols: {neg_issues if neg_issues else "None"}')

        # Feature columns align
        train_feat = set(train_df.columns)
        df_feat    = set(df.columns)
        missing_in_df   = train_feat - df_feat - {'Credit_Score'}
        extra_in_df     = df_feat - train_feat
        status = '✅' if not missing_in_df and not extra_in_df else '❌'
        print(f'  {status} Column alignment vs train: missing={missing_in_df}, extra={extra_in_df}')

    print('\n' + '=' * 55)


validate_prepared_data(train_set, valid_set, test_set)

FINAL DATA VALIDATION

--- Train ((80000, 41)) ---
  ✅ Missing values       : 0
  ⚠️  Non-numeric columns   : ['Occupation_Accountant', 'Occupation_Architect', 'Occupation_Developer', 'Occupation_Doctor', 'Occupation_Engineer', 'Occupation_Entrepreneur', 'Occupation_Journalist', 'Occupation_Lawyer', 'Occupation_Manager', 'Occupation_Mechanic', 'Occupation_Media_Manager', 'Occupation_Musician', 'Occupation_Scientist', 'Occupation_Teacher', 'Occupation_Writer', 'Payment_Behaviour_High_spent_Large_value_payments', 'Payment_Behaviour_High_spent_Medium_value_payments', 'Payment_Behaviour_High_spent_Small_value_payments', 'Payment_Behaviour_Low_spent_Large_value_payments', 'Payment_Behaviour_Low_spent_Medium_value_payments', 'Payment_Behaviour_Low_spent_Small_value_payments']
  ✅ Negative in non-neg cols: None
  ✅ Column alignment vs train: missing=set(), extra=set()

--- Valid ((20000, 41)) ---
  ✅ Missing values       : 0
  ⚠️  Non-numeric columns   : ['Occupation_Accountant', 'Occupation_

In [12]:
# Final overview of prepared data
print('Prepared train set — first 3 rows:')
train_set.head(3)

Prepared train set — first 3 rows:


,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Credit_Score,Occupation_Accountant,Occupation_Architect,Occupation_Developer,Occupation_Doctor,Occupation_Engineer,Occupation_Entrepreneur,Occupation_Journalist,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_Media_Manager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Payment_Behaviour_Low_spent_Small_value_payments
0,51.0000,101583.4800,3086.6833,5,7,10,4,8,8.0000,2.8900,5.0000,1,50.9300,34.4622,289.0000,0,190.8110,630.0158,314.0022,1,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False
1,23.0000,101926.9500,8635.9125,4,4,9,1,13,9.0000,10.2600,6.0000,1,1058.0000,39.6938,245.0000,0,70.5877,662.8039,410.1996,1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
2,49.0000,158871.1200,3086.6833,0,4,8,1,8,2.0000,1.1700,4.0000,2,576.4800,39.3672,228.0000,0,86.9059,746.8060,742.5142,1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False


## 11. Export Prepared Data <a id='11'></a>

In [13]:
def export_prepared_data(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    output_dir: str = r'../data/processed'
):
    """
    Export all three prepared splits to CSV files.

    Parameters
    ----------
    train_df   : Prepared training dataframe
    valid_df   : Prepared validation dataframe
    test_df    : Prepared test dataframe (no target column)
    output_dir : Directory path to save CSV files
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    train_path = f'{output_dir}/train_processed.csv'
    valid_path = f'{output_dir}/valid_processed.csv'
    test_path  = f'{output_dir}/test_processed.csv'

    train_df.to_csv(train_path, index=False)
    valid_df.to_csv(valid_path, index=False)
    test_df.to_csv(test_path,   index=False)

    print('✅ Exported prepared datasets:')
    print(f'   {train_path}  — {train_df.shape[0]:,} rows × {train_df.shape[1]} cols')
    print(f'   {valid_path}  — {valid_df.shape[0]:,} rows × {valid_df.shape[1]} cols')
    print(f'   {test_path}   — {test_df.shape[0]:,} rows × {test_df.shape[1]} cols')


export_prepared_data(train_set, valid_set, test_set)

✅ Exported prepared datasets:
   ../data/processed/train_processed.csv  — 80,000 rows × 41 cols
   ../data/processed/valid_processed.csv  — 20,000 rows × 41 cols
   ../data/processed/test_processed.csv   — 50,000 rows × 40 cols
